# Question 1: Word Segmentation and POS Tagging

This notebook implements and evaluates word segmentation (Trigram LM + Viterbi DP) and part-of-speech tagging (Trigram HMM + Viterbi DP) on English (Brown Corpus) and German (UD German-GSD), covering all 7 marking criteria from the assignment.


## Criterion 1: Train/Test Split & Data Handling (10 Marks)
- English: Brown Corpus with 80% train, 10% dev, 10% test split.
- German: Universal Dependencies German-GSD with official train, dev, and test splits.
- Extracted vocabulary sets and token frequencies.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('Q1'))

from data_handling import load_english_data, load_german_data, build_vocab, to_tagged_sents
from segmentation import TrigramLM, segment_viterbi
from tagger import HMMTagger, viterbi_tag
from morphology import convert_english_to_morph, convert_german_to_morph
from baselines import greedy_longest_match_segment, build_mft_baseline, tag_with_baseline
from evaluate import evaluate_pipeline, evaluate_baselines, confusion_matrix_to_df

# Load English Data (Brown corpus)
train_en, dev_en, test_en = load_english_data(split_ratio=0.8)
vocab_en, freq_en = build_vocab(train_en)
print(f"English Vocabulary Size: {len(vocab_en):,}")

# Load German Data (UD_German-GSD)
german_dir = 'UD_German-GSD' if os.path.exists('UD_German-GSD') else '../UD_German-GSD'
train_de_raw, dev_de_raw, test_de_raw = load_german_data(german_dir)
train_de = to_tagged_sents(train_de_raw, use_morph=False)
test_de = to_tagged_sents(test_de_raw, use_morph=False)
vocab_de, freq_de = build_vocab(train_de)
print(f"German Vocabulary Size: {len(vocab_de):,}")


## Criterion 2: Word Segmentation Model (Trigram LM + Viterbi DP) (15 Marks)
- Trigram Language Model trained on token sequences using Jelinek-Mercer linear interpolation smoothing.
- Character-level Viterbi dynamic programming decoder with beam search to find the optimal word boundary sequence.


In [ ]:
# Train English Trigram LM
lm_en = TrigramLM(train_sents=train_en, vocab=vocab_en)

# Test on Assignment Benchmark String
test_str_en = 'thequickbrownfoxjumpsoverthelazydog'
pred_words_en, score_en = segment_viterbi(test_str_en, lm_en)
print("Input:", test_str_en)
print("Predicted Segmentation:", pred_words_en)
print("Log Probability:", round(score_en, 2))


## Criterion 3: POS Tagging Model (Trigram HMM + Viterbi) (15 Marks)
- Second-order HMM learning emission probabilities P(w_i|t_i) and transition probabilities P(t_i|t_{i-2}, t_{i-1}).
- Viterbi decoding over segmented word candidates with candidate-tag pruning.


In [ ]:
# Train English HMM Tagger
hmm_en = HMMTagger(train_en)

# Tag the segmented words
pred_tagged_en = viterbi_tag(pred_words_en, hmm_en)
print("Segmented & Tagged Output:")
for word, tag in pred_tagged_en:
    print(f"  ({word}, {tag})")


## Criterion 4: Morphology-Aware Tagging Extension (15 Marks)
- English: Fine-grained Brown tags mapped into explicit morphological categories (Singular/Plural, Tense, Aspect, Degree).
- German: Universal POS tags extended with Gender and Number features to model agreement patterns.


In [ ]:
# English Morphology-Aware Model
train_en_morph = convert_english_to_morph(train_en)
test_en_morph = convert_english_to_morph(test_en)
hmm_en_morph = HMMTagger(train_en_morph)

print("Morphology-Aware English Output:")
pred_morph_en = viterbi_tag(pred_words_en, hmm_en_morph)
for word, tag in pred_morph_en:
    print(f"  ({word}, {tag})")


## Criterion 5: Baseline Models & Comparison (13 Marks)
- Segmentation Baseline: Greedy longest-match against training vocabulary.
- Tagging Baseline: Most-Frequent-Tag (MFT) with majority class fallback for unseen words.


In [ ]:
# Build Baselines
mft_en, default_tag_en = build_mft_baseline(train_en)

# Greedy Baseline on Test String
greedy_words_en = greedy_longest_match_segment(test_str_en, vocab_en)
print("Greedy Baseline Segmentation:", greedy_words_en)
print("MFT Baseline Tagging:", tag_with_baseline(greedy_words_en, mft_en, default_tag_en))


## Criterion 6: Comprehensive Evaluation & Error Analysis (15 Marks)
- Evaluates Word Segmentation and POS Tagging accuracy on test sets.
- Distinguishes between segmentation-caused errors and genuine tagging errors.
- Generates confusion matrix.


In [ ]:
# Evaluate English Proposed Pipeline vs Baselines (250 test sentences)
res_main_en = evaluate_pipeline(
    test_en,
    lambda txt: segment_viterbi(txt, lm_en)[0],
    lambda words: viterbi_tag(words, hmm_en),
    n_samples=250, seed=42
)

res_base_en = evaluate_baselines(
    test_en,
    lambda txt: greedy_longest_match_segment(txt, vocab_en),
    mft_en, default_tag_en,
    n_samples=250, seed=42
)

print(f"English Segmentation Accuracy (Model):    {res_main_en['seg_accuracy']:.2%}")
print(f"English Segmentation Accuracy (Baseline): {res_base_en['seg_accuracy']:.2%}")
print(f"Segmentation Gain:                        {res_main_en['seg_accuracy'] - res_base_en['seg_accuracy']:+.2%}")
print(f"English POS Tagging Accuracy (Model):     {res_main_en['tag_accuracy']:.2%}")
print(f"English POS Tagging Accuracy (Baseline):  {res_base_en['tag_accuracy']:.2%}")
print(f"Tagging Gain:                             {res_main_en['tag_accuracy'] - res_base_en['tag_accuracy']:+.2%}")

total_err_en = res_main_en['seg_caused_errors'] + res_main_en['genuine_tag_errors']
print(f"\nError Breakdown:")
print(f"  Segmentation-Caused Errors: {res_main_en['seg_caused_errors']} ({res_main_en['seg_caused_errors']/total_err_en:.1%})")
print(f"  Genuine Tagging Errors:     {res_main_en['genuine_tag_errors']} ({res_main_en['genuine_tag_errors']/total_err_en:.1%})")

# Display top 10x10 Confusion Matrix
cm_df_en = confusion_matrix_to_df(res_main_en['confusion_matrix'], hmm_en.tags, top_n=10)
print("\nTop Confusion Matrix:")
display(cm_df_en)


## German Language Pipeline (Morphologically Rich Comparison)
- Evaluates German UD-GSD treebank under identical methodology to compare with English.


In [ ]:
# Train German Trigram LM & HMM Taggers
lm_de = TrigramLM(train_de, vocab_de)
hmm_de = HMMTagger(train_de)

train_de_morph = convert_german_to_morph(train_de_raw)
test_de_morph = convert_german_to_morph(test_de_raw)
hmm_de_morph = HMMTagger(train_de_morph)

mft_de, default_tag_de = build_mft_baseline(train_de)

# Evaluate German
res_main_de = evaluate_pipeline(
    test_de,
    lambda txt: segment_viterbi(txt, lm_de, max_word_len=25)[0],
    lambda words: viterbi_tag(words, hmm_de),
    n_samples=250, seed=42
)

res_base_de = evaluate_baselines(
    test_de,
    lambda txt: greedy_longest_match_segment(txt, vocab_de, max_word_len=25),
    mft_de, default_tag_de,
    n_samples=250, seed=42
)

print(f"German Segmentation Accuracy (Model):    {res_main_de['seg_accuracy']:.2%}")
print(f"German Segmentation Accuracy (Baseline): {res_base_de['seg_accuracy']:.2%}")
print(f"German POS Tagging Accuracy (Model):     {res_main_de['tag_accuracy']:.2%}")
print(f"German POS Tagging Accuracy (Baseline):  {res_base_de['tag_accuracy']:.2%}")

# German Sample String (Compound Word)
test_str_de = 'autobahnmeistereiverwaltungsgebaeude'
words_de, _ = segment_viterbi(test_str_de, lm_de, max_word_len=25)
print(f"\nGerman Compound Input: {test_str_de}")
print(f"Segmented: {words_de}")
print(f"Standard POS: {viterbi_tag(words_de, hmm_de)}")
print(f"Morphology-Aware POS: {viterbi_tag(words_de, hmm_de_morph)}")


## Criterion 7: Comparative Analysis & Summary (17 Marks)
Side-by-side comparison of English vs. German across all dimensions.


In [ ]:
import pandas as pd

summary_df = pd.DataFrame({
    'Metric': [
        'Segmentation Accuracy (Trigram+DP)',
        'Segmentation Accuracy (Greedy Baseline)',
        'Segmentation Improvement (Delta)',
        'POS Tagging Accuracy (Trigram HMM)',
        'POS Tagging Accuracy (MFT Baseline)',
        'POS Tagging Improvement (Delta)',
        'Segmentation-Caused Errors (%)',
        'Genuine POS Tagging Errors (%)'
    ],
    'English (Brown)': [
        f"{res_main_en['seg_accuracy']:.2%}",
        f"{res_base_en['seg_accuracy']:.2%}",
        f"{(res_main_en['seg_accuracy'] - res_base_en['seg_accuracy']):+.2%}",
        f"{res_main_en['tag_accuracy']:.2%}",
        f"{res_base_en['tag_accuracy']:.2%}",
        f"{(res_main_en['tag_accuracy'] - res_base_en['tag_accuracy']):+.2%}",
        f"{(res_main_en['seg_caused_errors'] / max(1, res_main_en['seg_caused_errors'] + res_main_en['genuine_tag_errors'])):.1%}",
        f"{(res_main_en['genuine_tag_errors'] / max(1, res_main_en['seg_caused_errors'] + res_main_en['genuine_tag_errors'])):.1%}"
    ],
    'German (UD-GSD)': [
        f"{res_main_de['seg_accuracy']:.2%}",
        f"{res_base_de['seg_accuracy']:.2%}",
        f"{(res_main_de['seg_accuracy'] - res_base_de['seg_accuracy']):+.2%}",
        f"{res_main_de['tag_accuracy']:.2%}",
        f"{res_base_de['tag_accuracy']:.2%}",
        f"{(res_main_de['tag_accuracy'] - res_base_de['tag_accuracy']):+.2%}",
        f"{(res_main_de['seg_caused_errors'] / max(1, res_main_de['seg_caused_errors'] + res_main_de['genuine_tag_errors'])):.1%}",
        f"{(res_main_de['genuine_tag_errors'] / max(1, res_main_de['seg_caused_errors'] + res_main_de['genuine_tag_errors'])):.1%}"
    ]
})

display(summary_df)
